### Cell 1 — Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

# Feature Selection
from sklearn.feature_selection import chi2, f_classif, mutual_info_classif
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Multicollinearity
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Class Imbalance
from imblearn.combine import SMOTETomek

# Settings
os.chdir('/Users/cirrus/Desktop/PMOS_PROJECT/notebook')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

print('✅ All libraries imported successfully')

✅ All libraries imported successfully


### Cell 2 — Load Data from Block 1

In [3]:
df = pd.read_csv('../data/processed/pmos_eda_clean.csv')

print(f'✅ Data loaded: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'\nTarget distribution:')
print(df['PCOS (Y/N)'].value_counts())

✅ Data loaded: 541 rows × 46 columns

Target distribution:
PCOS (Y/N)
0    364
1    177
Name: count, dtype: int64


### Cell 3 — Clean Column Names

In [4]:
# Strip all leading/trailing spaces from column names
df.columns = df.columns.str.strip()

print('✅ Column names cleaned')
print('\nAll columns after cleaning:')
for i, col in enumerate(df.columns, 1):
    print(f'{i:2d}. "{col}"')

✅ Column names cleaned

All columns after cleaning:
 1. "Sl. No"
 2. "Patient File No."
 3. "PCOS (Y/N)"
 4. "Age (yrs)"
 5. "Weight (Kg)"
 6. "Height(Cm)"
 7. "BMI"
 8. "Blood Group"
 9. "Pulse rate(bpm)"
10. "RR (breaths/min)"
11. "Hb(g/dl)"
12. "Cycle(R/I)"
13. "Cycle length(days)"
14. "Marraige Status (Yrs)"
15. "Pregnant(Y/N)"
16. "No. of aborptions"
17. "I   beta-HCG(mIU/mL)"
18. "II    beta-HCG(mIU/mL)"
19. "FSH(mIU/mL)"
20. "LH(mIU/mL)"
21. "FSH/LH"
22. "Hip(inch)"
23. "Waist(inch)"
24. "Waist:Hip Ratio"
25. "TSH (mIU/L)"
26. "AMH(ng/mL)"
27. "PRL(ng/mL)"
28. "Vit D3 (ng/mL)"
29. "PRG(ng/mL)"
30. "RBS(mg/dl)"
31. "Weight gain(Y/N)"
32. "hair growth(Y/N)"
33. "Skin darkening (Y/N)"
34. "Hair loss(Y/N)"
35. "Pimples(Y/N)"
36. "Fast food (Y/N)"
37. "Reg.Exercise(Y/N)"
38. "BP _Systolic (mmHg)"
39. "BP _Diastolic (mmHg)"
40. "Follicle No. (L)"
41. "Follicle No. (R)"
42. "Avg. F size (L) (mm)"
43. "Avg. F size (R) (mm)"
44. "Endometrium (mm)"
45. "Unnamed: 44"
46. "LH_FSH_Ratio"


### Cell 4 — Fix Dtype Issues

In [5]:
# Check object columns
object_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Object columns found: {object_cols}')

# Convert to numeric
for col in object_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f'  ✅ {col} → converted to float | Missing created: {df[col].isnull().sum()}')

print(f'\nObject columns remaining: {df.select_dtypes(include="object").columns.tolist()}')

Object columns found: ['II    beta-HCG(mIU/mL)', 'Unnamed: 44']
  ✅ II    beta-HCG(mIU/mL) → converted to float | Missing created: 1
  ✅ Unnamed: 44 → converted to float | Missing created: 540

Object columns remaining: []


### Cell 5 — Impute Missing Values

In [6]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print('=== MISSING VALUES BEFORE IMPUTATION ===')
print(missing)

for col in missing.index:
    if df[col].nunique() <= 2:
        fill_val = df[col].mode()[0]
        strategy = 'mode'
    else:
        fill_val = df[col].median()
        strategy = 'median'
    
    df[col] = df[col].fillna(fill_val)
    print(f'  ✅ {col} → imputed with {strategy} ({fill_val:.3f})')

print(f'\nTotal missing values remaining: {df.isnull().sum().sum()}')

=== MISSING VALUES BEFORE IMPUTATION ===
Marraige Status (Yrs)       1
II    beta-HCG(mIU/mL)      1
AMH(ng/mL)                  1
Fast food (Y/N)             1
Unnamed: 44               540
dtype: int64
  ✅ Marraige Status (Yrs) → imputed with median (7.000)
  ✅ II    beta-HCG(mIU/mL) → imputed with median (1.990)
  ✅ AMH(ng/mL) → imputed with median (3.700)
  ✅ Fast food (Y/N) → imputed with mode (1.000)
  ✅ Unnamed: 44 → imputed with mode (7.000)

Total missing values remaining: 0


### Cell 6 —  Drop Irrelevant Columns

In [8]:
cols_to_drop = [
    'Sl. No',
    'Patient File No.',
    'Unnamed: 44',
    'Blood Group',
    'Pregnant(Y/N)',
]

# Only drop columns that exist
cols_to_drop = [col for col in cols_to_drop if col in df.columns]
df = df.drop(columns=cols_to_drop)

print(f'✅ Dropped {len(cols_to_drop)} columns')
print(f'   Dropped: {cols_to_drop}')
print(f'   Remaining shape: {df.shape}')

✅ Dropped 5 columns
   Dropped: ['Sl. No', 'Patient File No.', 'Unnamed: 44', 'Blood Group', 'Pregnant(Y/N)']
   Remaining shape: (541, 41)


### Cell 7 — Define Target and Features

In [9]:
target_col = 'PCOS (Y/N)'

X = df.drop(columns=[target_col])
y = df[target_col]

print(f'Features (X) : {X.shape}')
print(f'Target   (y) : {y.shape}')
print(f'\nFeature list:')
for i, col in enumerate(X.columns, 1):
    print(f'  {i:2d}. {col}')

Features (X) : (541, 40)
Target   (y) : (541,)

Feature list:
   1. Age (yrs)
   2. Weight (Kg)
   3. Height(Cm)
   4. BMI
   5. Pulse rate(bpm)
   6. RR (breaths/min)
   7. Hb(g/dl)
   8. Cycle(R/I)
   9. Cycle length(days)
  10. Marraige Status (Yrs)
  11. No. of aborptions
  12. I   beta-HCG(mIU/mL)
  13. II    beta-HCG(mIU/mL)
  14. FSH(mIU/mL)
  15. LH(mIU/mL)
  16. FSH/LH
  17. Hip(inch)
  18. Waist(inch)
  19. Waist:Hip Ratio
  20. TSH (mIU/L)
  21. AMH(ng/mL)
  22. PRL(ng/mL)
  23. Vit D3 (ng/mL)
  24. PRG(ng/mL)
  25. RBS(mg/dl)
  26. Weight gain(Y/N)
  27. hair growth(Y/N)
  28. Skin darkening (Y/N)
  29. Hair loss(Y/N)
  30. Pimples(Y/N)
  31. Fast food (Y/N)
  32. Reg.Exercise(Y/N)
  33. BP _Systolic (mmHg)
  34. BP _Diastolic (mmHg)
  35. Follicle No. (L)
  36. Follicle No. (R)
  37. Avg. F size (L) (mm)
  38. Avg. F size (R) (mm)
  39. Endometrium (mm)
  40. LH_FSH_Ratio
